# Forward Spectrum Summary

Scientific question: which sampled wavelength has the minimum reflectance, and how closely does the Python-defined O2 A case track the committed DISAMAR reference spectrum?

This notebook is an executable demo. It builds the DISAMAR O2 A reference case from the repo helper, calls the Python API, displays plots inline, and leaves interpretation tables in the executed notebook.

## Inputs and API Calls

The input is the Python-defined O2 A reference scene, including atmosphere, geometry, surface, aerosol, spectroscopy, CIA, and instrument-response controls. The API calls happen through a prepared O2 A context so diagnostics reuse the same resolved setup.

In [ ]:
from __future__ import annotations

import csv
import sys
import time
from dataclasses import asdict
from pathlib import Path

import altair as alt
import numpy as np
import pandas as pd

from validation.common.o2a_reference_case import build_o2a_case


def find_repo_root() -> Path:
    start = Path.cwd().resolve()
    for candidate in (start, *start.parents):
        if (candidate / "build.zig").exists() and (candidate / "pyproject.toml").exists():
            return candidate
    raise RuntimeError("could not find repository root")


REPO_ROOT = find_repo_root()
PYTHON_ROOT = REPO_ROOT / "python"
LIBRARY_NAME = "libzdisamar_c.dylib" if sys.platform == "darwin" else "libzdisamar_c.so"
LIBRARY_PATH = REPO_ROOT / "zig-out" / "lib" / LIBRARY_NAME
VENDOR_REFERENCE_PATH = REPO_ROOT / "validation" / "data" / "o2a_with_cia_disamar_reference.csv"
TOLERANCE = 1.0e-12

alt.data_transformers.disable_max_rows()


def require_library() -> str:
    if not LIBRARY_PATH.exists():
        raise FileNotFoundError(
            f"{LIBRARY_PATH} does not exist; build the native shared library first"
        )
    return str(LIBRARY_PATH)


def import_zdisamar():
    sys.path.insert(0, str(PYTHON_ROOT))
    import zdisamar as zd

    return zd


def load_spectrum_csv(path: Path) -> dict[str, np.ndarray]:
    with path.open(newline="") as handle:
        rows = list(csv.DictReader(handle))
    if not rows:
        raise ValueError(f"{path} is empty")
    return {key: np.array([float(row[key]) for row in rows], dtype=float) for key in rows[0]}


def interpolate_to_grid(
    wavelength_nm: np.ndarray,
    reference: dict[str, np.ndarray],
) -> dict[str, np.ndarray]:
    reference_wavelength_nm = reference["wavelength_nm"]
    if (
        wavelength_nm[0] < reference_wavelength_nm[0]
        or wavelength_nm[-1] > reference_wavelength_nm[-1]
    ):
        raise ValueError(
            "current spectrum grid extends outside the vendored DISAMAR reference grid"
        )
    return {
        key: wavelength_nm
        if key == "wavelength_nm"
        else np.interp(wavelength_nm, reference_wavelength_nm, values)
        for key, values in reference.items()
    }


def build_spectrum_chart(frame: pd.DataFrame, min_wavelength_nm: float) -> alt.Chart:
    long = frame.melt(
        id_vars=["wavelength_nm"],
        value_vars=[
            "zdisamar_reflectance",
            "reference_reflectance",
            "zdisamar_radiance",
            "reference_radiance",
            "zdisamar_irradiance",
            "reference_irradiance",
            "reflectance_residual",
        ],
        var_name="series",
        value_name="value",
    )
    quantities = {
        "zdisamar_reflectance": "reflectance",
        "reference_reflectance": "reflectance",
        "zdisamar_radiance": "radiance",
        "reference_radiance": "radiance",
        "zdisamar_irradiance": "irradiance",
        "reference_irradiance": "irradiance",
        "reflectance_residual": "reflectance residual",
    }
    long["quantity"] = long["series"].map(quantities)
    chart = (
        alt.Chart(long)
        .mark_line()
        .encode(
            x=alt.X("wavelength_nm:Q", title="wavelength (nm)"),
            y=alt.Y("value:Q", title=None),
            color=alt.Color("series:N", title=None),
            row=alt.Row("quantity:N", title=None),
            tooltip=["wavelength_nm:Q", "series:N", "value:Q"],
        )
        .properties(width=760, height=130)
        .resolve_scale(y="independent")
    )
    marker = (
        alt.Chart(pd.DataFrame({"wavelength_nm": [min_wavelength_nm]}))
        .mark_rule(color="#d62728", strokeDash=[4, 4])
        .encode(x="wavelength_nm:Q")
    )
    return chart + marker


def run_forward_summary() -> tuple[dict[str, object], alt.Chart]:
    library_path = require_library()
    zd = import_zdisamar()
    case = build_o2a_case(zd)

    total_start = time.perf_counter()
    prepare_start = time.perf_counter()
    with zd.prepare(case, library_path=library_path) as prepared:
        prepare_s = time.perf_counter() - prepare_start
        forward_start = time.perf_counter()
        with prepared.forward_model() as spectrum:
            forward_s = time.perf_counter() - forward_start
            report = spectrum.diagnostic_report
            wavelength_nm = spectrum.wavelength_nm.copy()
            radiance = spectrum.radiance.copy()
            irradiance = spectrum.irradiance.copy()
            reflectance = spectrum.reflectance.copy()

    min_index = int(np.argmin(reflectance))
    vendor = interpolate_to_grid(wavelength_nm, load_spectrum_csv(VENDOR_REFERENCE_PATH))
    reflectance_residual = reflectance - vendor["reflectance"]
    radiance_residual = radiance - vendor["radiance"]
    irradiance_residual = irradiance - vendor["irradiance"]
    frame = pd.DataFrame(
        {
            "wavelength_nm": wavelength_nm,
            "zdisamar_reflectance": reflectance,
            "reference_reflectance": vendor["reflectance"],
            "zdisamar_radiance": radiance,
            "reference_radiance": vendor["radiance"],
            "zdisamar_irradiance": irradiance,
            "reference_irradiance": vendor["irradiance"],
            "reflectance_residual": reflectance_residual,
        }
    )
    summary = {
        "sample_count": int(wavelength_nm.size),
        "min_reflectance": {
            "wavelength_nm": float(wavelength_nm[min_index]),
            "reflectance": float(reflectance[min_index]),
        },
        "diagnostic_report": asdict(report),
        "vendor_comparison": {
            "same_grid": bool(np.array_equal(wavelength_nm, vendor["wavelength_nm"])),
            "reflectance_mean_abs_residual": float(np.mean(np.abs(reflectance_residual))),
            "reflectance_max_abs_residual": float(np.max(np.abs(reflectance_residual))),
            "radiance_max_abs_residual": float(np.max(np.abs(radiance_residual))),
            "irradiance_max_abs_residual": float(np.max(np.abs(irradiance_residual))),
        },
        "timing": {
            "prepare_o2a_s": prepare_s,
            "forward_model_s": forward_s,
            "total_s": time.perf_counter() - total_start,
        },
    }
    return summary, build_spectrum_chart(frame, float(wavelength_nm[min_index]))

## Execute and Interpret

The cell below runs the demo and displays the comparison plot inline. Evaluate `summary` afterward to inspect the compact residual and timing report.

In [ ]:
summary, chart = run_forward_summary()
chart